# **Seq2Seq Model with Attention: Practice**

Hi, everyone! This practice is fully devoted to the Attention mechanism in RNNs. We will be working with a Seq2Seq model, a model made of two RNNs — an encoder and a decoder. In this practice you will:

- implement a simple Seq2Seq model from scratch;
- add an attention mechanism to the model;
- solve a machine translation task using pretrained models;
- analyse attention maps of pretrained models.

Happy coding!

![](https://ucarecdn.com/af53a4f7-2a0c-43b7-b444-dfff60077a33/)

**Make sure you clone the repository below!**

In [ ]:
!git clone https://github.com/SadSabrina/RNN_with_attention

In [ ]:
%cd RNN_with_attention/

In [ ]:
import tqdm
import matplotlib.pyplot as plt
import re
from PIL import Image
from matplotlib import ticker
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

## **Dataset**

We will be working with data collected by hard workers, volunteers and simply great people — sentences in two languages, Russian and English. The data is taken from [tatoeba.org](https://tatoeba.org/), where people keep adding new translations all the time.


The dataset is simple, two columns:
- `rus` — the sentence in Russian
- `eng` — the sentence in English

In [ ]:
# Load the dataset
total_df = pd.read_csv('data/rus_eng.csv')
total_df.head()

Already at this stage you can see that the dataset contains implicit duplicates caused by the different ways of saying the same thing in different languages. We will keep such sentences in the dataset. There may be other duplicates as well, but in order to optimise the process, we will look at them after the basic preprocessing.

**Question:** how many duplicates are there in the raw dataset?

In [ ]:
# Your code here

So let us do a basic preprocessing of every column and see whether duplicates appear after it.

In [ ]:
# Function for preprocessing the rows of the dataset

def normalizeString(s):
    s = s.lower().strip() #lowercase the string and trim the spaces at the edges
    s = re.sub(r"[^\w\s?]", "", s) # remove the punctuation marks, except the question mark
    s = re.sub(r"([?])", r" \1", s) # add a space before the question mark
    return s.strip() # trim the string at the edges once again

In [ ]:
total_df['rus'] = total_df['rus'].apply(lambda x: normalizeString(x))
total_df['eng'] = total_df['eng'].apply(lambda x: normalizeString(x))

In [ ]:
print('Duplicates in the dataset', total_df.duplicated().sum())

Since more duplicates have appeared, let us remove them all at once.

In [ ]:
total_df.drop_duplicates(inplace=True)

In [ ]:
total_df.info()

The cleaned dataset contains 724,991 values. **How many of them are duplicates in Russian? (not taking into account their different translations into English).**

In [ ]:
# Your code here

As you may have noticed, every operation on the dataset takes a small, but no longer instant amount of time. To speed up the data processing, we will work with a subset of it only.

In [ ]:
df_sample = total_df.sample(100000, random_state=42) # Do not change random_state

## Text for the model — turning the data into a form the network can "read"

To use words as inputs and targets, we need an encoding. We will implement it as a dictionary that maps every word to a number. For this, the class `Lang` with the dictionaries `word2index` and `index2word` is written. Besides that, let us reserve two indices for the special words (tokens) `SOS` and `EOS` — start and end of sequence respectively.

And in addition, we will keep a counter for every word inside the class.

In [ ]:
SOS_token = 0
EOS_token = 1

class Lang:
    def __init__(self):
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Counter of unique words (starting from 2, since there are SOS and EOS)

    def addSentences(self, sentences: pd.Series):
        """Adds all the sentences from a pandas Series."""
        sentences.apply(self.addSentence)

    def addSentence(self, sentence):
        """Adds a single sentence to the dictionary."""
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        """Adds a word to the dictionary or updates the counter if the word already exists."""
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

To turn the data into the relevant format before feeding it to the model, we need:
- a list of Russian-English pairs (where one language is the set of "features" and the other one is the set of "targets")
- a numeric representation of the words of both languages

To make the task easier for the model, let us filter the dataset - we will remove all the observations longer than 10 tokens. This will give us a chance to start getting interesting results in 40 minutes (Macbook M3 pro, on Colab the training will take longer).

In [ ]:
# Extracting the relevant columns

firtst_lang = df_sample['rus']
target_lang = df_sample['eng']

# Let us build pairs out of the columns
pairs = list(zip(firtst_lang, target_lang))

Let us get to the filtering.
Complete the function that implements the filter over the pairs. It has to "let a pair through" if both values in it are not longer than (less than or equal to) MAX_LENGTH and "block the pair" otherwise.

**Question: how many pairs did you get after the implementation? Check that the task is done correctly by entering the answer on Stepik.**

In [ ]:
MAX_LENGTH = 10

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and len(p[1].split(' ')) < MAX_LENGTH

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

# Let us run the filtering
pairs = filterPairs(pairs)

print("Filtered down to %s  pairs of sequences" % len(pairs))
print(random.choice(pairs))

In [ ]:
# Initialise the classes that store the languages
input_lang = Lang()
second_lang = Lang()

for pair in pairs:
     input_lang.addSentence(pair[0])
     second_lang.addSentence(pair[1])

print("Counted words in inputs and outputs:")
print(f'Input sentences contain {input_lang.n_words} unique values\n Output sentences contain {second_lang.n_words} unique values')

Note that `input` and `output langs` do not contain the words that occur in sequences longer than 10 and do not occur in the filtered ones. The model will not be able to work with them.

Now let us encode our sequences with indices and turn them into tensors.

In [ ]:
device = 'cpu'

def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')] # a sentence into a vector of indices

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(second_lang, pair[1])
    return (input_tensor, target_tensor)

**Look at the code below. What are the remaining values of the columns filled with, if the length of inp_ids < 10?**

In [ ]:
N = len(pairs)
BATCH_SIZE = 200

input_ids = np.zeros((N, MAX_LENGTH), dtype=np.int32) # blanks for storing our input indexes
target_ids = np.zeros((N, MAX_LENGTH), dtype=np.int32) # blanks for storing our target indexes

for idx, (inp, tgt) in enumerate(pairs):
    inp_ids = indexesFromSentence(input_lang, inp)
    inp_ids.append(EOS_token)

    tgt_ids = indexesFromSentence(second_lang, tgt)
    tgt_ids.append(EOS_token)

    input_ids[idx, :len(inp_ids)] = inp_ids #Filling the tensors with the corresponding values
    target_ids[idx, :len(tgt_ids)] = tgt_ids

train_data = TensorDataset(torch.LongTensor(input_ids), torch.LongTensor(target_ids))

train_sampler = RandomSampler(train_data)

train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=BATCH_SIZE)

## The Seq2Seq model

Let us prepare the model for solving the translation task. It will consist of two parts: the encoding one (encoder) and the decoding one (decoder).

A standard [Sequence to Sequence network](https://arxiv.org/abs/1409.3215) looks like this:

![](https://pytorch.org/tutorials/_static/img/seq-seq-images/seq2seq.png)

## Encoder-Decoder: base


The encoder of a seq2seq network is an RNN which, as a result of its transformations, outputs a vector and a hidden state for every word and uses the hidden state for
the next input word.


![](https://pytorch.org/tutorials/_static/img/seq-seq-images/encoder-network.png)

The decoder of a seq2seq network is another RNN which uses the output values of the encoder (the so-called context vector) and generates the output step by step.

![](https://pytorch.org/tutorials/_static/img/seq-seq-images/decoder-network.png)

Let us implement a simple encoder. It will consist of:
1. Initialisation of the embeddings (`nn.Embedding`);
2. Applying dropout;
3. A GRU layer.

Complete the initialisation of the embeddings and the GRU layer.

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = # Your code here — the layer that generates the embeddings'
        self.dropout = nn.Dropout(dropout_p)
        self.gru = # Your code here — the GRU layer


    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.gru(embedded)
        return output, hidden

**Implement a test encoder. What is the sum of the dimensions of the hidden states equal to?**

In [ ]:
test_encoder = EncoderRNN(2, 3)

test_tensor = torch.LongTensor(2)

test_encoder(test_tensor)[1].shape

## The motivation behind Attention

The idea of attention is simple — it lets the model relate different parts of the input and the output sequence. It is easy to picture why this matters in machine translation:
- Russian: "Zavtra ya poydu v kino."
- English: "I will go to the movies tomorrow."

## Attention — Bahdanau & Luong

There are many approaches to computing attention, for example Bahdanau-Attention, Luong-Attention, Self-attention and others. Besides, attention itself can be improved and enriched with information in different ways, through:
- The plain dot product: $e_i = \{(s_i, h_0), ..., (s_i, h_n)\} = [s_i h_j^T]_{j=0}^n$ — dot attention, aka Luong Attention
- Multiplicative attention with a weight matrix $W$: $e_i = [s_iWh_j^T]_{j=0}^n$, where $W$ is a learnable weight matrix (this turns out to be a generalisation of the dot product) - general attention, aka Luong Attention
- Bahdanau attention (MLP): $e_{ij} = tanh(h_jW_1 + s_iW_2)v$, where $W_1, W_2$ are learnable weight matrices, and $v$ is a learnable weight vector.

Besides, in all the examples above:
$(h_0, h_1, ..., h_n)$ are the hidden state vectors of the encoder
$(s_0 = h_n, s_1, ..., s_m)$ are the hidden state vectors of the decoder

In this tutorial we will implement Bahdanau attention and the first two modifications, called Luong Attention. In general, the notion of Luong Attention generalises the following ways:

- dot attention: $s_i^Th_j$ — we take the dot product of the hidden layers of the encoder and the decoder
- general attention: $s_i^TW_1h_j$ —  we multiply the hidden layers of the encoder and the decoder with an intermediate weight matrix between them
- concat attention: $v^Ttanh(W_1[s_i;h_j])$ — we multiply a learnable weight vector by the tangent of the weighted vector product of the hidden layers of the encoder and the decoder


You can take a look at the implementations of both mechanisms in the file `attentions.py`. Here we will look at the implementation of the Decoder with an attention mechanism and with [teacher forcing](https://en.wikipedia.org/wiki/Teacher_forcing#:~:text=Teacher%20forcing%20is%20an%20algorithm,to%20the%20ground%2Dtruth%20sequence.) applied.

In [ ]:
from attentions import BahdanauAttention, LuongAttention

class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1, method=None):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)

        if method:
            print(f'Luong Attention with {method} is used')
            self.attention = LuongAttention(method, hidden_size)

        else:
            print(f'Bahdanau Attention is used')
            self.attention = BahdanauAttention(hidden_size)

        self.gru = nn.GRU(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):         # we get the hidden and output state vectors of the encoder
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)    # we set an empty decoder input, filling it with SOS tokens
        decoder_hidden = encoder_hidden                                             # we set the dimension of the decoder hidden layer equal to the dimension of the encoder one
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: we feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: the standard approach to training
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions


    def forward_step(self, input, hidden, encoder_outputs):

        embedded =  self.dropout(self.embedding(input))

        decoder_hidden = hidden.permute(1, 0, 2)
        context, attn_weights = self.attention(decoder_hidden, encoder_outputs) # Note which entities attention relates to each other
        input_gru = torch.cat((embedded, context), dim=2)

        output, hidden = self.gru(input_gru, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

**Which of the entities produced and consumed by the models does attention relate to each other? Choose the answer on Stepik.**

# Training and evaluating the model.
The functions `train_epoch` and `train` are already written for this. You can simply import them from the file `helper.py`.

In [ ]:
from helper import train_epoch, train

hidden_size = 128

# Let us initialise one of the models LUONG DOT
# BAHDANAU
bahdanau_encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
bahdanau_decoder = AttnDecoderRNN(hidden_size, second_lang.n_words).to(device)

Now, to launch the training, it is enough to run the cell below. However, in order to save time, we will use pretrained models to get the maps.

In [ ]:
#bahdanau_losses, bahdanau_time = train(train_dataloader, bahdanau_encoder, bahdanau_decoder, 50, print_every=5)

In [ ]:
hidden_size = 128
device = 'cpu'

# LUONG DOT
dot_encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
dot_decoder = AttnDecoderRNN(hidden_size, second_lang.n_words, method='dot').to(device)

# LUONG GENERAL
general_encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
general_decoder = AttnDecoderRNN(hidden_size, second_lang.n_words, method='general').to(device)

# LUONG CONCAT
concat_encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
concat_decoder = AttnDecoderRNN(hidden_size, second_lang.n_words, method='concat').to(device)

# BAHDANAU
bahdanau_encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
bahdanau_decoder = AttnDecoderRNN(hidden_size, second_lang.n_words).to(device)

In [ ]:
# DOT ATTENTION
dot_encoder.load_state_dict(torch.load('models/dot_encoder', weights_only=True))
dot_encoder.eval()

dot_decoder.load_state_dict(torch.load('models/dot_decoder', weights_only=True))
dot_decoder.eval()

In [ ]:
# GENERAL
general_encoder.load_state_dict(torch.load('models/general_encoder', weights_only=True))
general_encoder.eval()

general_decoder.load_state_dict(torch.load('models/general_decoder', weights_only=True))
general_decoder.eval()

In [ ]:
# CONCAT
concat_encoder.load_state_dict(torch.load('models/concat_encoder', weights_only=True))
concat_encoder.eval()

concat_decoder.load_state_dict(torch.load('models/concat_decoder', weights_only=True))
concat_decoder.eval()

In [ ]:
# BAHDANAU
bahdanau_encoder.load_state_dict(torch.load('models/bahdanau_encoder', weights_only=True))
bahdanau_encoder.eval()

bahdanau_decoder.load_state_dict(torch.load('models/bahdanau_decoder', weights_only=True))
bahdanau_decoder.eval()

## Visualising the attentions.
Let us take the sentence "ya lyublyu koshek" ("I love cats"). And let us see how models trained under identical conditions, but with different attention mechanisms, cope with it.

In [ ]:
encoders_decoders_pairs = [(dot_encoder, dot_decoder), (concat_encoder, concat_decoder), (general_encoder, general_decoder), (bahdanau_encoder, bahdanau_decoder)]
keys = ['DOT', 'CONCAT', 'GENERAL', 'BAHDANAU']

In [ ]:
# Function for getting the prediction of the model (1)

def evaluate(encoder, decoder, sentence, input_lang, output_lang): # We get the encoder, the decoder, the sentence, and also the input and output dictionaries (Lang)
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence) # we turn the sentence into a tensor, using the words in input_lang

        encoder_outputs, encoder_hidden = encoder(input_tensor) # we feed the sentence to the encoder
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden) # we feed the sentence to the decoder

        _, topi = decoder_outputs.topk(1) # we get the prediction
        decoded_ids = topi.squeeze()

        decoded_words = []    # We decode the words
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('EOS')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [ ]:
# Function for visualising the attentions (2)

def showAttention(input_sentence, output_words, attentions): # we get the sentence, the words and the attentions

    fig = plt.figure(figsize=(20, 8))
    ax = fig.add_subplot(111)
    cax = ax.matshow(attentions.cpu().numpy(), cmap='bone') # we draw the weights as a matrix
    fig.colorbar(cax)

    # We set up the names on the cells
    ax.set_xticklabels([''] + input_sentence.split(' ') +
                       ['EOS'], rotation=90)
    ax.set_yticklabels([''] + output_words)

    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

# We combine functions 1 and 2, and also save the result

def evaluateAndShowAttention(input_sentence, title, encoder, decoder): # we get the sentence, the title of the plot, the encoder and the decoder

    output_words, attentions = evaluate(encoder, decoder, input_sentence, input_lang, second_lang) # We get the words and the attentions
    print('input =', input_sentence)
    print('output =', ' '.join(output_words))
    showAttention(input_sentence, output_words, attentions[0, :len(output_words), :]) # we visualise the resulting matrix
    plt.title(title)

    plt.savefig(f'{title}.png') # we save the plot



for enc_dec, key in zip(encoders_decoders_pairs, keys):
    enc, dec = enc_dec
    evaluateAndShowAttention('ya lyublyu koshek', key, enc, dec)

In [ ]:
# Let us load the numeric results of the training
train_results_data = pd.read_csv('results_data.csv',
                                 index_col=0)

train_results_data

In [ ]:
img = Image.open('results/Losses.png')

img

**Compare the attention maps you got and analyse them, together with the data about the training of the models. Write down your conclusions and send them as a free-form essay on Stepik.**